In [1]:
from datasets import load_dataset

ds = load_dataset("Genius-Society/Pima")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [6]:
!pip install pennylane datasets
import pennylane as qml
from pennylane import numpy as np
from datasets import load_dataset
train_ds = ds["train"]

X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'], s['SkinThickness'],
               s['Insulin'], s['BMI'], s['DiabetesPedigreeFunction'], s['Age']]
              for s in train_ds ])
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
y = np.array([s['Outcome'] for s in ds['train']])
num_qubits = X.shape[1]
dev = qml.device("default.qubit", wires=num_qubits)

In [3]:
# QSample Encoding
def qsample_encoding(x):
    for i in range(len(x)):
        qml.RY(np.arcsin(np.sqrt(x[i])), wires=i)

@qml.qnode(dev)
def circuit_qsample(x):
    qsample_encoding(x)
    return [qml.expval(qml.PauliZ(i)) for i in range(len(x))]

# Exemple pour la première ligne
result = circuit_qsample(X_norm[0])

# Convertir en valeurs classiques
result_numpy = np.array([v.item() for v in result])

print("QSample Encoding (classical values):")
print(result_numpy)

QSample Encoding (classical values):
[0.87447463 0.42502372 0.73720978 0.77919372 0.91133355 0.61926985
 0.91173647 0.94971616]


In [4]:
# On prépare le tableau pour stocker les résultats
X_qsample = []

print("Encodage QSample en cours...")
for row in X_norm:
    # On récupère les 8 valeurs d'espérance (une par qubit)
    res = circuit_qsample(row)
    X_qsample.append([v.item() for v in res])

X_qsample = np.array(X_qsample)

print(f"Forme des données QSample : {X_qsample.shape}") # (768, 8)

Encodage QSample en cours...
Forme des données QSample : (614, 8)


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Split
X_train_qs, X_test_qs, y_train, y_test = train_test_split(X_qsample, y, test_size=0.2, random_state=42)

# Modèle
clf_qs = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_qs.fit(X_train_qs, y_train)

# Score
y_pred_qs = clf_qs.predict(X_test_qs)
print(f"Précision avec QSample Encoding : {accuracy_score(y_test, y_pred_qs):.2%}")

Précision avec QSample Encoding : 74.80%
